# Feature Engineering — Telco Churn

Verification wrapper for the feature discovery and feature engineering pipeline.
The engineering lives in `sql/features/` and `src/telco_churn/features/`; this
notebook loads the built feature view and confirms the output shape and schema.

**Feature discovery outcome:** nine candidates tested; one adopted — `charge_per_service`
(`monthlycharges ÷ active service count`). Eight candidates failed at least one of
four gate screens (leakage, redundancy, ΔPR-AUC ≥ +0.0015, permutation importance).
Full discovery narrative: `notebooks/02a-feature-discovery.ipynb`.

**Feature set entering model training:** 19 raw IBM columns + `charge_per_service` = **20 columns**.

## Prerequisites

This notebook reads live data from a local Postgres database loaded by the data ingestion pipeline.

Before running:
1. Start Postgres: `docker compose up postgres -d`
2. Add `POSTGRES_URL` to `.env` at the project root
3. Ingest raw data: `uv run python -m telco_churn.data.ingest`

In [1]:
from __future__ import annotations

import warnings
from pathlib import Path
import os

if not Path("configs").exists():
    os.chdir("..")

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dotenv import load_dotenv
from omegaconf import OmegaConf

from telco_churn.features import (
    FEATURE_SCHEMA,
    SQL_FEATURE_COLS,
    build_feature_df,
    build_sql_features,
)
from telco_churn.utils.db import get_engine
from telco_churn.utils.paths import get_project_root

load_dotenv()
cfg = OmegaConf.load("configs/config.yaml")
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

## Load Feature Data

In [2]:
try:
    engine = get_engine()
    build_sql_features(engine, sql_dir=Path(cfg.paths.sql_features))
except Exception as e:
    raise RuntimeError(
        "Cannot reach Postgres. Is the container running? See Prerequisites above."
    ) from e

df_raw = pd.read_sql_table("customer_features", engine, columns=SQL_FEATURE_COLS)
df = build_feature_df(df_raw)

feature_cols = list(FEATURE_SCHEMA.binary + FEATURE_SCHEMA.multi_cat + FEATURE_SCHEMA.numeric)
print(f"Loaded {len(df):,} rows")
print(f"Feature columns: {len(feature_cols)} ({len(FEATURE_SCHEMA.binary)} binary, {len(FEATURE_SCHEMA.multi_cat)} multi-cat, {len(FEATURE_SCHEMA.numeric)} numeric)")
print(f"Adopted SQL-engineered column present: {'charge_per_service' in df.columns}")
df[["customerid", "tenure", "monthlycharges", "charge_per_service", "churn"]].round({"charge_per_service": 2}).head(5)

2026-06-29 08:46:31 [info     ] feature view created           duration_ms=54 file=charge_per_service.sql


2026-06-29 08:46:31 [info     ] feature view created           duration_ms=38 file=customer_features.sql


Loaded 7,043 rows
Feature columns: 20 (6 binary, 10 multi-cat, 4 numeric)
Adopted SQL-engineered column present: True


,customerid,tenure,monthlycharges,charge_per_service,churn
0,8665-UTDHZ,1,30.20,15.10,1
1,5248-YGIJN,72,90.25,10.03,0
2,8773-HHUOZ,17,64.70,16.18,1
3,3841-NFECX,71,96.35,13.76,0
4,4929-XIHVW,2,95.50,19.10,0


### Feature Inventory

In [3]:
rows = [
    ("Binary",        FEATURE_SCHEMA.binary),
    ("Multi-category",FEATURE_SCHEMA.multi_cat),
    ("Numeric",       FEATURE_SCHEMA.numeric),
]
inventory = pd.DataFrame(
    [(group, len(cols), ", ".join(f"`{c}`" for c in cols)) for group, cols in rows],
    columns=["Group", "Count", "Columns"],
)
total = pd.DataFrame([["**Total**", f"**{sum(len(c) for _, c in rows)}**", ""]],
                     columns=inventory.columns)
pd.concat([inventory, total], ignore_index=True).style.hide(axis="index")

Group,Count,Columns
Binary,6,"`gender`, `has_partner`, `dependents`, `phoneservice`, `paperlessbilling`, `seniorcitizen`"
Multi-category,10,"`multiplelines`, `internetservice`, `onlinesecurity`, `onlinebackup`, `deviceprotection`, `techsupport`, `streamingtv`, `streamingmovies`, `contract_type`, `paymentmethod`"
Numeric,4,"`tenure`, `monthlycharges`, `totalcharges`, `charge_per_service`"
**Total**,**20**,


## Summary

This notebook builds the SQL feature views, loads the `customer_features` DataFrame from Postgres, and renders the 20-column feature inventory (6 binary, 10 multi-category, 4 numeric) that enters model training. Logic lives in `sql/features/` and `src/telco_churn/features/`; this notebook calls those functions and confirms the output shape and schema.

For the full engineering narrative — the adopted feature (`charge_per_service`), its construction, and the final feature set composition — see **[§3b Feature Engineering](../ANALYSIS.md#3b-feature-engineering)** in `ANALYSIS.md`. For the discovery process that produced the adopted set, see `notebooks/02a-feature-discovery.ipynb`.